# Looping the network: `network_loops` × `loop_layers` on typed-decisions

sysone 0.7 adds two settings to the backbone:

- `network_loops` is **how many times the top layers run**, with the same weights: more depth per
  decision, not more parameters. Each pass reads the previous pass's output.
- `loop_layers` is **which layers loop**: the top K kept layers, as a count or a fraction of the
  layers (default 0.25).

On Ettin-encoder-17m (7 layers), looping the top 2 layers 3 times raised test accuracy from about
0.56 to about 0.67, on both an Apple GPU and CPU (seed 0). Looping 2 layers twice did almost
nothing, and looping 4 layers 3 times did less well than 2 layers 3 times: a small block looped
several times worked best. This notebook checks whether that carries over to larger models, on
[typed-decisions](https://huggingface.co/datasets/LocalLLaMA/typed-decisions), in two steps:

| step | what | runs | expected time, Colab L4 |
|---|---|---|---|
| 1 | Ettin-encoder-150m (22 layers): no loops, top 25% × 2/3/4, top 10% × 3/4 | 6 per seed | 20–35 min each, about 2.5–3 hours per seed |
| 2 | Ettin-encoder-400m (28 layers): no loops and step 1's winner | 2 per seed | about 1–1.5 hours each |

It runs **one seed** (`SEEDS = [0]`). If a setting leads the no-loop model by about a point or more,
set `SEEDS = [0, 1, 2]` and run again: finished runs are read back from Drive, so only the new
seeds train. One seed can't confirm a small gain.

Every extra pass costs the looped layers' compute again, in training and in prediction, so each
run also records how long training and predicting the 400 test records took.

> **⚠️ Use a GPU runtime** (Runtime → Change runtime type → L4 or A100) and connect Google Drive
> when asked. Every run autosaves to Drive and writes its results there when it finishes. If the
> runtime disconnects, run the notebook again from the top: finished runs are read back, and an
> unfinished one continues from its last save. You can stop after any run; the summaries use
> whatever has finished.

In [ ]:
# sysone 0.7 isn't on PyPI yet: this installs it from its development branch. Restart the
# runtime first if an older sysone is installed in this session.
import importlib.util

SYSONE = "sysone @ git+https://github.com/ForwardMetric/sysone.git@milestone-7-network-loops"
if importlib.util.find_spec("sysone") is None:
    %pip install -q "$SYSONE"
if importlib.util.find_spec("datasets") is None:
    %pip install -q datasets

In [ ]:
import json
import math
import statistics
import time
from pathlib import Path

import pandas as pd
import torch
from datasets import load_dataset

import sysone
from sysone import SystemOneModel

assert "network_loops" in SystemOneModel().get_params(), "this needs sysone 0.7: restart the runtime and run again"
print("sysone", sysone.__version__)

# Results and checkpoints go to Google Drive on Colab, so a disconnect loses nothing; elsewhere
# they go to a local folder.
try:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/sysone-network-loops")
except ImportError:
    OUT = Path("sysone-network-loops")
OUT.mkdir(parents=True, exist_ok=True)
print("saving to", OUT)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (this will be very slow)")

## Load the data

All four workflows: 1,200 training and 400 test records, five questions each, with soft labels
(each answer is the raters' probability spread).

In [ ]:
def load_rows(split):
    ds = load_dataset("LocalLLaMA/typed-decisions", "all", split=split)
    return [{"state": r["state"], "questions": r["questions"], "gold": r["gold"], "category": r["workflow"]}
            for r in ds]


train = load_rows("train")
test = load_rows("test")
print("%d training records, %d test records" % (len(train), len(test)))

## Scoring, question by question

The leaderboard's formulas (from the dataset's `bench/score_with_jobs.py`): accuracy is the
predicted top option matching the raters' top option (a tie goes to the first option), plus KL
from the raters' distribution, Brier summed over options and ECE in 10 bins.

Every test question is also kept with its type and the **raters' margin**, the gap between their
top two answers, so results can be broken down: a gain on questions the raters clearly agree on
means something different from one on near-ties.

In [ ]:
QUESTION_TYPES = {}  # (record index, question id) -> type
for i, r in enumerate(test):
    for qid, q in json.loads(r["questions"]).items():
        QUESTION_TYPES[(i, qid)] = q["type"]


def score_questions(model, rows):
    """One record per test question: type, raters' margin, hit, KL, Brier and confidence."""
    results = model.predict([{"state": r["state"], "questions": r["questions"]} for r in rows])
    out = []
    for i, (row, result) in enumerate(zip(rows, results)):
        for qid, gold in json.loads(row["gold"]).items():
            answer = result["answers"][qid]
            probs = answer.get("probabilities") or {"false": 1 - answer["noul"], "true": answer["noul"]}
            options = list(gold["probabilities"])
            g = [float(gold["probabilities"][o]) for o in options]
            p = [float(probs.get(o, 0.0)) for o in options]
            total = sum(p)
            p = [v / total for v in p] if total > 0 else [1 / len(p)] * len(p)
            top = sorted(g, reverse=True)
            out.append({
                "record": i, "question": qid, "type": QUESTION_TYPES[(i, qid)],
                "margin": top[0] - (top[1] if len(top) > 1 else 0.0),
                "hit": p.index(max(p)) == g.index(max(g)),
                "kl": sum(gi * math.log(gi / max(pi, 1e-9)) for gi, pi in zip(g, p) if gi > 0),
                "brier": sum((pi - gi) ** 2 for pi, gi in zip(p, g)),
                "confidence": max(p),
            })
    return out


def leaderboard(questions):
    n = len(questions)
    bins = [[0, 0, 0.0] for _ in range(10)]  # count, hits, confidence sum
    for q in questions:
        b = bins[min(int(q["confidence"] * 10), 9)]
        b[0] += 1
        b[1] += q["hit"]
        b[2] += q["confidence"]
    return {"accuracy": sum(q["hit"] for q in questions) / n,
            "kl_from_gold": sum(q["kl"] for q in questions) / n,
            "brier": sum(q["brier"] for q in questions) / n,
            "ece": sum(t / n * abs(h / t - c / t) for t, h, c in bins if t)}

## Train, score and save, resumably

`run` trains one model and writes to `OUT/<name>/`: autosaves in `checkpoint/`, and
`results.json` with the leaderboard scores, every test question's result, and the time to train
and to predict the test set. The model itself isn't kept (many runs would fill a Drive); add
`model.save(...)` if you want one.

In [ ]:
def run(name, model_name, **params):
    folder = OUT / name
    results_file = folder / "results.json"
    if results_file.exists():
        print("%s: already done" % name)
        return json.loads(results_file.read_text())

    checkpoint = folder / "checkpoint"
    start = time.perf_counter()
    if (checkpoint / "run.json").exists():
        info = json.loads((checkpoint / "run.json").read_text())
        print("%s: resuming from step %d (%s)" % (name, info["step"], info["reason"]))
        model = SystemOneModel.load(checkpoint)
        if info["reason"] != "finished":
            model.fit(train)
    else:
        print("%s: training" % name)
        model = SystemOneModel(model_name, checkpoint_dir=str(checkpoint), **params)
        model.fit(train)
    fit_minutes = (time.perf_counter() - start) / 60

    start = time.perf_counter()
    questions = score_questions(model, test)
    predict_seconds = time.perf_counter() - start
    results = {"name": name, "model_name": model_name, "params": params, "fit_minutes": fit_minutes,
               "predict_seconds": predict_seconds, "leaderboard": leaderboard(questions), "questions": questions}
    results_file.write_text(json.dumps(results, default=str))
    print("%s: leaderboard accuracy %.4f, %.1f minutes" % (name, results["leaderboard"]["accuracy"], fit_minutes))
    return results


# Settings shared by every run. Validation runs about ten times per epoch; training stops once
# validation loss hasn't improved for 3 epochs and keeps the best weights.
COMMON = dict(tuning="full", epochs=10, batch_size=32, checkpoint_every=300)
# (loop_layers, network_loops); (0, 1) is the model without loops
SETTINGS = [(0, 1), (0.25, 2), (0.25, 3), (0.25, 4), (0.1, 3), (0.1, 4)]
SEEDS = [0]  # one seed first; add 1 and 2 to confirm a gain (finished runs are kept)


def label(k, loops):
    return "no loops" if loops == 1 else "top %d%% × %d" % (round(100 * k), loops)


def loop_params(k, loops):
    return {} if loops == 1 else {"network_loops": loops, "loop_layers": k}


def run_name(size, k, loops, seed):
    return "ettin-%s-%s-seed%d" % (size, "noloop" if loops == 1 else "top%d-x%d" % (round(100 * k), loops), seed)

## Step 1: Ettin-encoder-150m

22 layers, 768 wide, like ModernBERT-base, so it uses ModernBERT-base's measured learning rate
(5e-5). Top 25% is the top 6 layers, top 10% the top 2. Runs go seed by seed, so after the first
six you already have one full set to look at.

In [ ]:
step1 = {}
for seed in SEEDS:
    for k, loops in SETTINGS:
        step1[(k, loops, seed)] = run(run_name("150m", k, loops, seed), "jhu-clsp/ettin-encoder-150m",
                                      backbone_lr=5e-5, seed=seed, **loop_params(k, loops), **COMMON)

### Results

Mean and spread across seeds, and the **paired difference**: each setting minus the no-loop model
on the same seed, which removes most of the seed-to-seed noise. A gain that is smaller than its
spread across seeds isn't one yet.

In [ ]:
def settings_in(results):
    """The settings in `results`, no loops first."""
    return sorted({(k, l) for k, l, _ in results}, key=lambda s: (s[1] > 1, s[0], s[1]))


def summarize(results):
    rows = {}
    for k, loops in settings_in(results):
        seeds = [s for kk, l, s in results if (kk, l) == (k, loops)]
        acc = [results[(k, loops, s)]["leaderboard"]["accuracy"] for s in seeds]
        diffs = [results[(k, loops, s)]["leaderboard"]["accuracy"] - results[(0, 1, s)]["leaderboard"]["accuracy"]
                 for s in seeds if (0, 1, s) in results]
        rows[label(k, loops)] = {
            "seeds": len(seeds),
            "accuracy": statistics.mean(acc),
            "accuracy sd": statistics.stdev(acc) if len(acc) > 1 else float("nan"),
            "vs no loops": statistics.mean(diffs) if diffs else float("nan"),
            "diff sd": statistics.stdev(diffs) if len(diffs) > 1 else float("nan"),
            **{m: statistics.mean(results[(k, loops, s)]["leaderboard"][m] for s in seeds)
               for m in ("kl_from_gold", "brier", "ece")},
            "train min": statistics.mean(results[(k, loops, s)]["fit_minutes"] for s in seeds),
            "predict s": statistics.mean(results[(k, loops, s)]["predict_seconds"] for s in seeds),
        }
    return pd.DataFrame.from_dict(rows, orient="index")


summarize(step1).round(4)

### Where the gains come from

Accuracy by the raters' margin (near-ties, where even raters split, versus clear questions) and by
question type, averaged over seeds.

In [ ]:
BUCKETS = [(0.0, 0.1, "near-tie (≤0.1)"), (0.1, 0.3, "middle (0.1–0.3)"), (0.3, 1.01, "clear (>0.3)")]


def breakdown(results, by):
    rows = {}
    for k, loops in settings_in(results):
        qs = [q for kk, l, s in results if (kk, l) == (k, loops) for q in results[(kk, l, s)]["questions"]]
        if by == "margin":
            groups = {name: [q for q in qs if lo <= q["margin"] < hi] for lo, hi, name in BUCKETS}
        else:
            groups = {}
            for q in qs:
                groups.setdefault(q["type"], []).append(q)
        rows[label(k, loops)] = {name: sum(q["hit"] for q in g) / len(g) for name, g in groups.items() if g}
    return pd.DataFrame.from_dict(rows, orient="index")


display(breakdown(step1, "margin").round(4))
display(breakdown(step1, "type").round(4))

## Step 2: confirm on Ettin-encoder-400m

Set `WINNER` to step 1's best `(loop_layers, network_loops)`. This trains it and the model without
loops, at the learning rate of the earlier 0.8025 run (3e-5). The earlier best on 400m was 0.8025;
the leaderboard's best fine-tuned entries are 4B models at about 0.82.

Only adopt the winner if it gains about a point or more over the no-loop model on average across
seeds, with a spread smaller than the gain.

In [ ]:
WINNER = (0.1, 3)  # step 1's best (loop_layers, network_loops)

step2 = {}
for seed in SEEDS:
    for k, loops in [(0, 1), WINNER]:
        step2[(k, loops, seed)] = run(run_name("400m", k, loops, seed), "jhu-clsp/ettin-encoder-400m",
                                      backbone_lr=3e-5, seed=seed, **loop_params(k, loops), **COMMON)

display(summarize(step2).round(4))
display(breakdown(step2, "margin").round(4))

## Reading the results

- About 18% of test questions have the raters' top two answers within 10 points, so no model will
  get all of those right; gains there are worth less than gains on the clear questions.
- KL and Brier show whether the probabilities, not just the top answer, got closer to the raters'.
- "predict s" is the time to answer all 400 test records (2,000 questions); compare it with the
  no-loop model to see what the extra depth costs at serving time.